<a href="https://colab.research.google.com/github/HonoreT/Eve-Marie_metagenomic-BITS-16S/blob/main/notebooks/Eve-Marie_Analyses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Marie-Eve_Analysis")
RAW = PROJECT / "02_raw_fastq"

BITS = RAW / "BITS"
EMP  = RAW / "EMP"

print("PROJECT existe :", PROJECT.exists())
print("BITS existe    :", BITS.exists())
print("EMP existe     :", EMP.exists())

PROJECT existe : True
BITS existe    : True
EMP existe     : True


In [ ]:
from pathlib import Path
import pandas as pd
import re

records = []

for dataset, folder in [("BITS", BITS), ("EMP", EMP)]:
    for f in sorted(folder.glob("*.fastq.gz")):

        name = f.name

        read = None
        if re.search(r"_R1_", name):
            read = "R1"
        elif re.search(r"_R2_", name):
            read = "R2"

        records.append({
            "dataset": dataset,
            "filename": name,
            "read": read,
            "size_MB": round(f.stat().st_size / 1024**2, 2),
            "path": str(f)
        })

inventory = pd.DataFrame(records)

print(inventory.to_string(index=False))

print("\nNombre total de fichiers :", len(inventory))
print("\nNombre par dataset :")
print(inventory.groupby("dataset").size())

print("\nNombre R1/R2 :")
print(inventory.groupby(["dataset", "read"]).size())

dataset                                filename read  size_MB                                                                                                path
   BITS 10BITSLEM26-200_S1_L001_R1_001.fastq.gz   R1     4.25 /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/10BITSLEM26-200_S1_L001_R1_001.fastq.gz
   BITS 10BITSLEM26-200_S1_L001_R2_001.fastq.gz   R2    10.33 /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/10BITSLEM26-200_S1_L001_R2_001.fastq.gz
   BITS 11BITSLEM26-200_S1_L001_R1_001.fastq.gz   R1     4.48 /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/11BITSLEM26-200_S1_L001_R1_001.fastq.gz
   BITS 11BITSLEM26-200_S1_L001_R2_001.fastq.gz   R2    10.50 /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/11BITSLEM26-200_S1_L001_R2_001.fastq.gz
   BITS 12BITSLEM26-200_S1_L001_R1_001.fastq.gz   R1     6.34 /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/12BITSLEM26-200_S1_L001_R1_001.fastq.gz
   BITS 12BITSLEM26-200_S1_L

In [ ]:
def sample_from_filename(name):
    return re.sub(r"_R[12]_001\.fastq\.gz$", "", name)

inventory["fastq_sample"] = inventory["filename"].apply(sample_from_filename)

pair_check = (
    inventory
    .groupby(["dataset", "fastq_sample"])["read"]
    .nunique()
    .reset_index(name="n_reads")
)

print(pair_check.to_string(index=False))

print("\nÉchantillons sans paire R1/R2 complète :")
print(pair_check[pair_check["n_reads"] != 2])

dataset            fastq_sample  n_reads
   BITS 10BITSLEM26-200_S1_L001        2
   BITS 11BITSLEM26-200_S1_L001        2
   BITS 12BITSLEM26-200_S1_L001        2
   BITS 13BITSLEM26-200_S1_L001        2
   BITS 14BITSLEM26-200_S1_L001        2
   BITS 15BITSLEM26-200_S1_L001        2
   BITS 16BITSLEM26-200_S1_L001        2
   BITS 17BITSLEM26-200_S1_L001        2
   BITS 18BITSLEM26-200_S1_L001        2
   BITS 19BITSLEM26-200_S1_L001        2
   BITS  1BITSLEM26-200_S1_L001        2
   BITS 20BITSLEM26-200_S1_L001        2
   BITS 21BITSLEM26-200_S1_L001        2
   BITS 22BITSLEM26-200_S1_L001        2
   BITS 23BITSLEM26-200_S1_L001        2
   BITS 24BITSLEM26-200_S1_L001        2
   BITS 25BITSLEM26-200_S1_L001        2
   BITS 26BITSLEM26-200_S1_L001        2
   BITS 27BITSLEM26-200_S1_L001        2
   BITS 28BITSLEM26-200_S1_L001        2
   BITS 29BITSLEM26-200_S1_L001        2
   BITS  2BITSLEM26-200_S1_L001        2
   BITS 30BITSLEM26-200_S1_L001        2
   BITS 31BITSLE

In [ ]:
import gzip

bad_files = []

for f in inventory["path"]:
    try:
        with gzip.open(f, "rb") as handle:
            while handle.read(1024 * 1024):
                pass
    except Exception as e:
        bad_files.append((f, str(e)))

if len(bad_files) == 0:
    print("OK : tous les fichiers .fastq.gz testés sont lisibles.")
else:
    print("Fichiers problématiques :")
    for x in bad_files:
        print(x)

OK : tous les fichiers .fastq.gz testés sont lisibles.


In [ ]:
import gzip

for dataset, folder in [("BITS", BITS), ("EMP", EMP)]:

    files = sorted(folder.glob("*_R1_*.fastq.gz"))

    print("\n" + "=" * 80)
    print(dataset)
    print("=" * 80)

    if files:
        f = files[0]
        print("Fichier :", f.name)

        with gzip.open(f, "rt") as handle:
            for i in range(3):
                header = handle.readline().strip()
                seq    = handle.readline().strip()
                plus   = handle.readline().strip()
                qual   = handle.readline().strip()

                print("\nRead", i + 1)
                print(header)
                print(seq)
                print("Longueur :", len(seq))


BITS
Fichier : 10BITSLEM26-200_S1_L001_R1_001.fastq.gz

Read 1
@AV234504:MetaG_WGS_202604:2541403811:1:10801:0563:0111 1:N:0:TAATGCGC+TTCTAGCT
ACCTGCGGAGGGATCATTAAAGAGTTGCAAAACTCCCAAACCATTGTGAACATACCTTCAACGTTGCTTCGGCGGGTTGGCCCCGGTCTCCGGGGTCCCCGGCCCTACTCGGGCGCCCGCCGGAGGTATCTAACTCTTGAACTTTTATGGCCTCTCTGAGTCTTTGTACTTAATAAGTCAAAACTTTCAGCAACGGATCTCAGATCGGAAGAGCACACGTCTGAACTCCAGTCACTAATGCGCATCTCGTATGCCGTCTTCTGCTTGATACGGCGACCACCGAGATCTACACTTCTAGCT
Longueur : 300

Read 2
@AV234504:MetaG_WGS_202604:2541403811:1:10801:0021:0121 1:N:0:TAATGCGC+TTCTAGCT
ACCTGCGGAGGGATCATTACCGAGTGAGGGCTCCGGCCCGACCTCCGACCCCAAACCTTCCAACCCCCTCGTACCTGGGGGGTGACCCGGCCGCGCGCCGGGGCCCCCACGGGACCTCTCCAACCCTGTCTTATCGAGCGTCTGAGCGTAAAAGCGAAATCAAACAAAACTTTCAACAATGGATCTCAGATCGGAAGAGCACAGTCTGAACTCCAGTCACTAATGCGCATCTCGTATGCCGTCTTCTGCTTGATTCGGCGACCACCGAGATCTACACTTCTAGCTACCCTCTTTCCCTAC
Longueur : 300

Read 3
@AV234504:MetaG_WGS_202604:2541403811:1:10801:0491:0120 1:N:0:TAATGCGC+TTCTAGCT
ACCTGCGGAGGGATCATTACCGAGTGAGGGCCCTTTGGGTCCAACCTC

In [ ]:
import gzip
from collections import Counter

def first_bases_summary(folder, pattern, n_reads=1000, n_bases=30):
    files = sorted(folder.glob(pattern))

    if not files:
        print(f"Aucun fichier trouvé : {pattern}")
        return

    f = files[0]

    prefixes = []

    with gzip.open(f, "rt") as handle:
        for i in range(n_reads):
            header = handle.readline()
            seq    = handle.readline().strip()
            plus   = handle.readline()
            qual   = handle.readline()

            if not seq:
                break

            prefixes.append(seq[:n_bases])

    counts = Counter(prefixes)

    print("\n" + "="*80)
    print("Fichier :", f.name)
    print("Reads examinés :", len(prefixes))
    print(f"Top 15 des {n_bases} premiers nucléotides")
    print("="*80)

    for seq, count in counts.most_common(15):
        print(count, seq)


for dataset, folder in [("BITS", BITS), ("EMP", EMP)]:

    print("\n\n############################")
    print(dataset, "- R1")
    print("############################")
    first_bases_summary(folder, "*_R1_*.fastq.gz")

    print("\n############################")
    print(dataset, "- R2")
    print("############################")
    first_bases_summary(folder, "*_R2_*.fastq.gz")



############################
BITS - R1
############################

Fichier : 10BITSLEM26-200_S1_L001_R1_001.fastq.gz
Reads examinés : 1000
Top 15 des 30 premiers nucléotides
169 ACCTGCGGAGGGATCATTACCGAGTGAGGG
103 ACCTGCGGAGGGATCATTAAAAGAAGCCGA
77 ACCTGCGGAAGGATCATTACCGAGTGAGGG
71 ACCTGCGGAGGGATCATTACAGAGTTGCAA
49 ACCTGCGGAAGGATCATTAAAAGAAGCCGA
46 ACCTGCGGAGGGATCATTACAAGAAGCCGA
43 ACCTGCGGAAGGATCATTACAGAGTTGCAA
41 ACCTGCGGAGGGATCATTACCGAGTTTACA
27 ACCTGCGGAAGGATCATCACTGAGATGTGG
26 ACCTGCGGAGGGATCATTACCGAGTTATTA
19 ACCTGCGGAGGGATCATCACTGAGATGTGG
18 ACCTGCGGAGGGATCATTATCCATCTCAAC
16 ACCTGCGGAGGGATCATTACCGAAGTTACT
15 ACCTGCGGAGGGATCATTCATAATAAGTGT
14 ACCTGCGGAAGGATCATTACAAGAAGCCGA

############################
BITS - R2
############################

Fichier : 10BITSLEM26-200_S1_L001_R2_001.fastq.gz
Reads examinés : 1000
Top 15 des 30 premiers nucléotides
42 GAGATCCGTTGTTGAAAGTTTTAACTTATT
39 GAGATCCGTTGCTGAAAGTTTTGACTTATT
33 GAGATCCGTTGTTGAAAGTTTTGACTTATT
31 GAGATCCATTGCTGAAAGTTTTAACTGA

In [ ]:
import gzip
from collections import Counter

def show_prefixes(folder, pattern, n_reads=1000, n_bases=25):

    files = sorted(folder.glob(pattern))
    f = files[0]

    prefixes = []

    with gzip.open(f, "rt") as handle:

        for i in range(n_reads):

            header = handle.readline()
            seq = handle.readline().strip()
            handle.readline()
            handle.readline()

            if not seq:
                break

            prefixes.append(seq[:n_bases])

    counts = Counter(prefixes)

    print("\nFichier :", f.name)
    print("Reads :", len(prefixes))

    for seq, n in counts.most_common(10):
        print(f"{n:4d}  {seq}")


print("========== BITS R1 ==========")
show_prefixes(BITS, "*_R1_*.fastq.gz")

print("\n========== BITS R2 ==========")
show_prefixes(BITS, "*_R2_*.fastq.gz")

========== BITS R1 ==========

Fichier : 10BITSLEM26-200_S1_L001_R1_001.fastq.gz
Reads : 1000
 243  ACCTGCGGAGGGATCATTACCGAGT
 103  ACCTGCGGAGGGATCATTAAAAGAA
  96  ACCTGCGGAAGGATCATTACCGAGT
  78  ACCTGCGGAGGGATCATTACAGAGT
  49  ACCTGCGGAAGGATCATTAAAAGAA
  47  ACCTGCGGAAGGATCATTACAGAGT
  46  ACCTGCGGAGGGATCATTACAAGAA
  27  ACCTGCGGAAGGATCATCACTGAGA
  20  ACCTGCGGAGGGATCATTAGTGAAT
  19  ACCTGCGGAGGGATCATCACTGAGA

========== BITS R2 ==========

Fichier : 10BITSLEM26-200_S1_L001_R2_001.fastq.gz
Reads : 1000
  70  GAGATCCGTTGTTGAAAGTTTTAAC
  62  GAGATCCATTGTTGAAAGTTTTAAC
  61  GAGATCCGTTGCTGAAAGTTTTAAC
  54  GAGATCCATTGCTGAAAGTTTTAAC
  42  GAGATCCGTTGCTGAAAGTTTTGAC
  41  GAGATCCGTTGTTGAAAGTTTTGAC
  34  GAGATCCATTGCTGAAAGTTTTGAC
  33  GAGATCCATTGTTAAAAGTTTTAAC
  30  GAGATCCGTTGTTAAAAGTTTTAAC
  29  GAGATCCATTGTTGAAAGTTTTGAC


In [ ]:
from pathlib import Path
import re
import pandas as pd

def extract_sample_number(filename, dataset):
    """
    Exemples attendus :
    1BITSLEM26-200_S1_L001_R1_001.fastq.gz
    1EMPLEM26-200_S1_L001_R1_001.fastq.gz
    """
    pattern = rf"^(\d+){dataset}LEM26-200_"
    m = re.match(pattern, filename)

    if m:
        return int(m.group(1))
    return None


records = []

for dataset, folder in [("BITS", BITS), ("EMP", EMP)]:

    for f in sorted(folder.glob("*.fastq.gz")):

        read = (
            "R1" if "_R1_" in f.name
            else "R2" if "_R2_" in f.name
            else "?"
        )

        sample_no = extract_sample_number(f.name, dataset)

        records.append({
            "dataset": dataset,
            "sample_no": sample_no,
            "read": read,
            "filename": f.name
        })


df = pd.DataFrame(records)

print("Nombre total de fichiers :", len(df))

print("\nFichiers par dataset :")
print(df.groupby("dataset").size())

print("\nNombre de R1/R2 :")
print(df.groupby(["dataset", "read"]).size())


print("\n===== Vérification des échantillons 1 à 48 =====")

expected = set(range(1, 49))

for dataset in ["BITS", "EMP"]:

    observed = set(
        df.loc[df["dataset"] == dataset, "sample_no"]
        .dropna()
        .astype(int)
    )

    missing = sorted(expected - observed)
    extra   = sorted(observed - expected)

    print(f"\n{dataset}")
    print("Nombre d'échantillons :", len(observed))
    print("Manquants :", missing)
    print("Supplémentaires :", extra)


print("\n===== Vérification R1/R2 par échantillon =====")

pairs = (
    df.groupby(["dataset", "sample_no"])["read"]
      .agg(lambda x: sorted(set(x)))
      .reset_index()
)

problems = pairs[pairs["read"].apply(lambda x: x != ["R1", "R2"])]

if problems.empty:
    print("✅ Tous les échantillons possèdent exactement R1 et R2.")
else:
    print("⚠️ Problèmes détectés :")
    print(problems.to_string(index=False))

Nombre total de fichiers : 192

Fichiers par dataset :
dataset
BITS    96
EMP     96
dtype: int64

Nombre de R1/R2 :
dataset  read
BITS     R1      48
         R2      48
EMP      R1      48
         R2      48
dtype: int64

===== Vérification des échantillons 1 à 48 =====

BITS
Nombre d'échantillons : 48
Manquants : []
Supplémentaires : []

EMP
Nombre d'échantillons : 48
Manquants : []
Supplémentaires : []

===== Vérification R1/R2 par échantillon =====
✅ Tous les échantillons possèdent exactement R1 et R2.


In [ ]:
import pandas as pd

metadata_path = PROJECT / "01_metadata" / "Metadata_EM.xlsx"

meta = pd.read_excel(metadata_path)

print("Colonnes du fichier de métadonnées :")
print(meta.columns.tolist())

print("\nAperçu :")
print(meta.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx'

In [ ]:
from pathlib import Path

metadata_dir = PROJECT / "01_metadata"

print("Dossier recherché :", metadata_dir)
print("Existe :", metadata_dir.exists())

print("\nContenu du dossier :")

if metadata_dir.exists():
    files = list(metadata_dir.iterdir())

    if files:
        for f in files:
            print(f.name)
    else:
        print("⚠️ Le dossier existe, mais il est vide.")
else:
    print("❌ Le dossier 01_metadata n'existe pas.")

Dossier recherché : /content/drive/MyDrive/Marie-Eve_Analysis/01_metadata
Existe : True

Contenu du dossier :
⚠️ Le dossier existe, mais il est vide.


In [ ]:
metadata_path = PROJECT / "01_metadata" / "Metadata_EM.xlsx"
meta = pd.read_excel(metadata_path)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx'

In [ ]:
import pandas as pd

metadata_path = PROJECT / "01_metadata" / "Metadata_EM.xlsx"

print("Fichier existe :", metadata_path.exists())
print("Chemin :", metadata_path)

meta = pd.read_excel(metadata_path)

print("\nColonnes du fichier de métadonnées :")
print(meta.columns.tolist())

print("\nDimensions :", meta.shape)

print("\nAperçu :")
print(meta.head())

Fichier existe : True
Chemin : /content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx

Colonnes du fichier de métadonnées :
['Échantillon No. ', 'Bloc', 'Cultivar', 'Traitement', 'pH', 'CE', 'NH4', 'NO3', 'P', 'K', 'Ca', 'Mg', 'Fe', 'Cu', 'Mn', 'Zn', 'Mo', 'Al']

Dimensions : (48, 18)

Aperçu :
   Échantillon No.   Bloc Cultivar Traitement    pH     CE     NH4     NO3  \
0                 1     1  Blueray    CTRLMin  3.51  350.3    4.78   0.000   
1                 2     1  Blueray     CTROrg  5.66    469  119.14  12.609   
2                 3     1  Blueray    PGPROrg  3.75  869.9   17.47   2.826   
3                 4     1  Blueray    PGPRMin  2.90  790.7    8.24   0.000   
4                 5     1     Duke    CTRLMin  2.89  687.1    6.06   0.000   

        P        K       Ca       Mg      Fe    Cu     Mn     Zn    Mo  \
0  318.10   539.83   898.98   386.24  516.86  1.95  12.58  11.24  4.21   
1  313.70   240.29  4320.88   460.74  475.25  2.51  11.10  13.86  3.74 

In [ ]:
# ============================================
# Vérification finale FASTQ <-> métadonnées
# ============================================

# Nettoyer les noms de colonnes
meta.columns = meta.columns.str.strip()

print("Colonnes après nettoyage :")
print(meta.columns.tolist())

# Identifier la colonne échantillon
sample_col = "Échantillon No."

# Convertir en entier
meta[sample_col] = pd.to_numeric(meta[sample_col], errors="coerce")

print("\nNombre de lignes metadata :", len(meta))
print("Nombre d'identifiants uniques :", meta[sample_col].nunique())

# Détecter valeurs manquantes ou dupliquées
missing_ids = meta[meta[sample_col].isna()]
duplicates = meta[meta[sample_col].duplicated(keep=False)]

print("\nIdentifiants manquants dans metadata :", len(missing_ids))
print("Identifiants dupliqués dans metadata :", len(duplicates))

# Ensemble attendu d'après les metadata
metadata_ids = set(meta[sample_col].dropna().astype(int))

print("\nPlage des identifiants metadata :",
      min(metadata_ids), "à", max(metadata_ids))

# Comparaison avec chaque jeu FASTQ
for dataset in ["BITS", "EMP"]:

    fastq_ids = set(
        df.loc[df["dataset"] == dataset, "sample_no"]
        .dropna()
        .astype(int)
    )

    missing_in_fastq = sorted(metadata_ids - fastq_ids)
    extra_in_fastq   = sorted(fastq_ids - metadata_ids)

    print("\n" + "="*50)
    print(dataset)
    print("="*50)

    print("IDs metadata :", len(metadata_ids))
    print("IDs FASTQ    :", len(fastq_ids))
    print("Présents dans metadata mais absents FASTQ :", missing_in_fastq)
    print("Présents dans FASTQ mais absents metadata :", extra_in_fastq)

    if not missing_in_fastq and not extra_in_fastq:
        print("✅ Correspondance parfaite FASTQ <-> metadata")
    else:
        print("⚠️ Incohérence détectée")

# Vérification attendue 1 à 48
expected_ids = set(range(1, 49))

print("\n" + "="*50)
print("Vérification finale 1 à 48")
print("="*50)

print("IDs manquants dans metadata :", sorted(expected_ids - metadata_ids))
print("IDs supplémentaires :", sorted(metadata_ids - expected_ids))

if metadata_ids == expected_ids:
    print("✅ Les métadonnées contiennent exactement les échantillons 1 à 48.")

Colonnes après nettoyage :
['Échantillon No.', 'Bloc', 'Cultivar', 'Traitement', 'pH', 'CE', 'NH4', 'NO3', 'P', 'K', 'Ca', 'Mg', 'Fe', 'Cu', 'Mn', 'Zn', 'Mo', 'Al']

Nombre de lignes metadata : 48
Nombre d'identifiants uniques : 48

Identifiants manquants dans metadata : 0
Identifiants dupliqués dans metadata : 0

Plage des identifiants metadata : 1 à 48

BITS
IDs metadata : 48
IDs FASTQ    : 48
Présents dans metadata mais absents FASTQ : []
Présents dans FASTQ mais absents metadata : []
✅ Correspondance parfaite FASTQ <-> metadata

EMP
IDs metadata : 48
IDs FASTQ    : 48
Présents dans metadata mais absents FASTQ : []
Présents dans FASTQ mais absents metadata : []
✅ Correspondance parfaite FASTQ <-> metadata

Vérification finale 1 à 48
IDs manquants dans metadata : []
IDs supplémentaires : []
✅ Les métadonnées contiennent exactement les échantillons 1 à 48.


In [ ]:
# ============================================
# ÉTAPE 2A — Installation FastQC + MultiQC
# ============================================

!apt-get update -qq
!apt-get install -y -qq fastqc
!pip install -q multiqc

print("FastQC :")
!fastqc --version

print("\nMultiQC :")
!multiqc --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package at-spi2-common.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../01-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1build1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0

In [ ]:
# ============================================
# Création des dossiers QC
# ============================================

QC_DIR = PROJECT / "04_work" / "fastqc"
QC_BITS = QC_DIR / "BITS"
QC_EMP  = QC_DIR / "EMP"

QC_BITS.mkdir(parents=True, exist_ok=True)
QC_EMP.mkdir(parents=True, exist_ok=True)

print("BITS QC :", QC_BITS)
print("EMP QC  :", QC_EMP)

BITS QC : /content/drive/MyDrive/Marie-Eve_Analysis/04_work/fastqc/BITS
EMP QC  : /content/drive/MyDrive/Marie-Eve_Analysis/04_work/fastqc/EMP


In [ ]:
# ============================================
# FastQC - BITS
# ============================================

!fastqc \
    -t 4 \
    -o "{QC_BITS}" \
    "{BITS}"/*.fastq.gz

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
applicat

In [ ]:
# Vérifier le nombre de rapports générés

from pathlib import Path

html_bits = list(QC_BITS.glob("*_fastqc.html"))
zip_bits  = list(QC_BITS.glob("*_fastqc.zip"))

print("Rapports HTML BITS :", len(html_bits))
print("Archives ZIP BITS  :", len(zip_bits))

Rapports HTML BITS : 96
Archives ZIP BITS  : 96


In [ ]:
# ============================================
# ÉTAPE 2D — MultiQC pour BITS
# ============================================

BITS_MULTIQC = PROJECT / "05_results" / "multiqc_BITS"
BITS_MULTIQC.mkdir(parents=True, exist_ok=True)

!multiqc "{QC_BITS}" \
    -o "{BITS_MULTIQC}" \
    --force


/// ]8;id=735507;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /content/drive/MyDrive/Marie-Eve_Analysis/04_work/fastqc/BITS
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 192/192  
            fastqc | Found 96 reports
     write_results | Data        : drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_BITS/multiqc_data
     write_results | Report      : drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_BITS/multiqc_report.html
           multiqc | MultiQC complete


In [ ]:
print("Contenu du dossier MultiQC BITS :")

for f in sorted(BITS_MULTIQC.iterdir()):
    print(f.name)

Contenu du dossier MultiQC BITS :
multiqc_data
multiqc_report.html


In [ ]:
from IPython.display import HTML, display

report_path = BITS_MULTIQC / "multiqc_report.html"

display(HTML(filename=str(report_path)))

In [ ]:
print(report_path)

/content/drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_BITS/multiqc_report.html


In [ ]:
import zipfile, os, json, pandas as pd, re, textwrap, glob, pathlib
zip_path="/mnt/data/multiqc_BITS-20261007T153330Z-1-001.zip"
with zipfile.ZipFile(zip_path) as z:
    names=z.namelist()
names[:50], len(names)


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/multiqc_BITS-20261007T153330Z-1-001.zip'

In [ ]:
# ============================================
# ÉTAPE 2E — FastQC EMP
# ============================================

!fastqc \
    -t 4 \
    -o "{QC_EMP}" \
    "{EMP}"/*.fastq.gz

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
applicat

In [ ]:
html_emp = list(QC_EMP.glob("*_fastqc.html"))
zip_emp  = list(QC_EMP.glob("*_fastqc.zip"))

print("Rapports HTML EMP :", len(html_emp))
print("Archives ZIP EMP  :", len(zip_emp))

Rapports HTML EMP : 96
Archives ZIP EMP  : 96


In [ ]:
# ============================================
# ÉTAPE 2F — MultiQC pour EMP
# ============================================

EMP_MULTIQC = PROJECT / "05_results" / "multiqc_EMP"
EMP_MULTIQC.mkdir(parents=True, exist_ok=True)

!multiqc "{QC_EMP}" \
    -o "{EMP_MULTIQC}" \
    --force


/// ]8;id=798792;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /content/drive/MyDrive/Marie-Eve_Analysis/04_work/fastqc/EMP
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 192/192  
            fastqc | Found 96 reports
     write_results | Data        : drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_EMP/multiqc_data
     write_results | Report      : drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_EMP/multiqc_report.html
           multiqc | MultiQC complete


In [ ]:
print("Contenu du dossier MultiQC EMP :")

for f in sorted(EMP_MULTIQC.iterdir()):
    print(f.name)

Contenu du dossier MultiQC EMP :
multiqc_data
multiqc_report.html


In [ ]:
!pip install -q cutadapt

print("Version Cutadapt :")
!cutadapt --version

KeyboardInterrupt: 

In [ ]:
EMP_FWD = "GTGYCAGCMGCCGCGGTAA"
EMP_REV = "GGACTACNVGGGTWTCTAAT"

print("EMP forward primer :", EMP_FWD)
print("EMP reverse primer :", EMP_REV)

EMP forward primer : GTGYCAGCMGCCGCGGTAA
EMP reverse primer : GGACTACNVGGGTWTCTAAT


In [ ]:
EMP_TRIM = PROJECT / "04_work" / "cutadapt_EMP"
EMP_TRIM.mkdir(parents=True, exist_ok=True)

test_r1 = EMP / "1EMPLEM26-200_S1_L001_R1_001.fastq.gz"
test_r2 = EMP / "1EMPLEM26-200_S1_L001_R2_001.fastq.gz"

out_r1 = EMP_TRIM / "1EMP_R1_trimmed.fastq.gz"
out_r2 = EMP_TRIM / "1EMP_R2_trimmed.fastq.gz"

!cutadapt \
    -g "{EMP_FWD}" \
    -G "{EMP_REV}" \
    -o "{out_r1}" \
    -p "{out_r2}" \
    "{test_r1}" "{test_r2}"

This is cutadapt 5.2 with Python 3.13.16
Command line parameters: -g GTGYCAGCMGCCGCGGTAA -G GGACTACNVGGGTWTCTAAT -o /content/drive/MyDrive/Marie-Eve_Analysis/04_work/cutadapt_EMP/1EMP_R1_trimmed.fastq.gz -p /content/drive/MyDrive/Marie-Eve_Analysis/04_work/cutadapt_EMP/1EMP_R2_trimmed.fastq.gz /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/EMP/1EMPLEM26-200_S1_L001_R1_001.fastq.gz /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/EMP/1EMPLEM26-200_S1_L001_R2_001.fastq.gz
Processing paired-end reads on 1 core ...
Done           00:00:03        78,374 reads @  39.5 µs/read;   1.52 M reads/minute

=== Summary ===

Total read pairs processed:             78,374
  Read 1 with adapter:                  77,951 (99.5%)
  Read 2 with adapter:                  78,080 (99.6%)
Pairs written (passing filters):        78,374 (100.0%)

Total basepairs processed:    47,024,394 bp
  Read 1:    23,512,197 bp
  Read 2:    23,512,197 bp
Total written (filtered):     43,982,990 bp (93.5%)
  Re

In [ ]:
Total read pairs processed
Read 1 with adapter
Read 2 with adapter
Pairs written

SyntaxError: invalid syntax (2324606056.py, line 1)

In [ ]:
import re

EMP_TRIM_ALL = PROJECT / "04_work" / "cutadapt_EMP_all"
EMP_TRIM_ALL.mkdir(parents=True, exist_ok=True)

r1_files = sorted(EMP.glob("*_R1_*.fastq.gz"))

for r1 in r1_files:

    r2 = Path(str(r1).replace("_R1_", "_R2_"))

    sample = re.sub(r"_R1_001\.fastq\.gz$", "", r1.name)

    out1 = EMP_TRIM_ALL / f"{sample}_R1_trimmed.fastq.gz"
    out2 = EMP_TRIM_ALL / f"{sample}_R2_trimmed.fastq.gz"

    !cutadapt \
        -g "{EMP_FWD}" \
        -G "{EMP_REV}" \
        -o "{out1}" \
        -p "{out2}" \
        "{r1}" "{r2}"

This is cutadapt 5.2 with Python 3.13.16
Command line parameters: -g GTGYCAGCMGCCGCGGTAA -G GGACTACNVGGGTWTCTAAT -o /content/drive/MyDrive/Marie-Eve_Analysis/04_work/cutadapt_EMP_all/10EMPLEM26-200_S1_L001_R1_trimmed.fastq.gz -p /content/drive/MyDrive/Marie-Eve_Analysis/04_work/cutadapt_EMP_all/10EMPLEM26-200_S1_L001_R2_trimmed.fastq.gz /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/EMP/10EMPLEM26-200_S1_L001_R1_001.fastq.gz /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/EMP/10EMPLEM26-200_S1_L001_R2_001.fastq.gz
Processing paired-end reads on 1 core ...
Done           00:00:03        84,699 reads @  42.0 µs/read;   1.43 M reads/minute

=== Summary ===

Total read pairs processed:             84,699
  Read 1 with adapter:                  84,474 (99.7%)
  Read 2 with adapter:                  84,243 (99.5%)
Pairs written (passing filters):        84,699 (100.0%)

Total basepairs processed:    50,819,398 bp
  Read 1:    25,409,699 bp
  Read 2:    25,409,699 bp
Total writ

In [ ]:
trimmed_emp = list(EMP_TRIM_ALL.glob("*.fastq.gz"))

print("Nombre de fichiers EMP trimmed :", len(trimmed_emp))

Nombre de fichiers EMP trimmed : 96


In [ ]:
QC_EMP_TRIM = PROJECT / "04_work" / "fastqc_EMP_trimmed"
QC_EMP_TRIM.mkdir(parents=True, exist_ok=True)

!fastqc \
    -t 4 \
    -o "{QC_EMP_TRIM}" \
    "{EMP_TRIM_ALL}"/*.fastq.gz

[0.005s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.005s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
applicat

In [ ]:
EMP_TRIM_MULTIQC = PROJECT / "05_results" / "multiqc_EMP_trimmed"
EMP_TRIM_MULTIQC.mkdir(parents=True, exist_ok=True)

!multiqc "{QC_EMP_TRIM}" \
    -o "{EMP_TRIM_MULTIQC}" \
    --force


/// ]8;id=540331;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /content/drive/MyDrive/Marie-Eve_Analysis/04_work/fastqc_EMP_trimmed
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 192/192  
            fastqc | Found 96 reports
     write_results | Data        : drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_EMP_trimmed/multiqc_data
     write_results | Report      : drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_EMP_trimmed/multiqc_report.html
           multiqc | MultiQC complete


In [ ]:
import pandas as pd
from pathlib import Path
import re

trimmed_dir = EMP_TRIM_ALL

records = []

for sample_no in range(1, 49):

    prefix = f"{sample_no}EMPLEM26-200_S1_L001"

    r1 = trimmed_dir / f"{prefix}_R1_trimmed.fastq.gz"
    r2 = trimmed_dir / f"{prefix}_R2_trimmed.fastq.gz"

    records.append({
        "sample-id": str(sample_no),
        "forward-absolute-filepath": str(r1),
        "reverse-absolute-filepath": str(r2)
    })

manifest_emp = pd.DataFrame(records)

print(manifest_emp.head())
print("\nNombre d'échantillons :", len(manifest_emp))

print("\nFichiers R1 manquants :",
      sum(~manifest_emp["forward-absolute-filepath"].apply(lambda x: Path(x).exists())))

print("Fichiers R2 manquants :",
      sum(~manifest_emp["reverse-absolute-filepath"].apply(lambda x: Path(x).exists())))

  sample-id                          forward-absolute-filepath  \
0         1  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...   
1         2  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...   
2         3  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...   
3         4  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...   
4         5  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...   

                           reverse-absolute-filepath  
0  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...  
1  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...  
2  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...  
3  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...  
4  /content/drive/MyDrive/Marie-Eve_Analysis/04_w...  

Nombre d'échantillons : 48

Fichiers R1 manquants : 0
Fichiers R2 manquants : 0


In [ ]:
inventory_path = PROJECT / "05_results" / "EMP_trimmed_inventory.tsv"

manifest_emp.to_csv(
    inventory_path,
    sep="\t",
    index=False
)

print("Inventaire sauvegardé :", inventory_path)

Inventaire sauvegardé : /content/drive/MyDrive/Marie-Eve_Analysis/05_results/EMP_trimmed_inventory.tsv


In [ ]:
import pandas as pd
from pathlib import Path

src = Path("/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx")
out = Path("/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM_qiime2.tsv")

df = pd.read_excel(src)

print("Shape initiale :", df.shape)
print("Colonnes :", list(df.columns))

df = df.rename(columns={"Échantillon No.": "sample-id"})
df["sample-id"] = df["sample-id"].astype(str).str.strip()

df.to_csv(out, sep="\t", index=False)

print("\nFichier créé :", out)
print("Nombre de lignes :", len(df))
print("Nombre d'IDs uniques :", df["sample-id"].nunique())
print("Doublons :", df["sample-id"].duplicated().sum())

print("\nAperçu :")
print(df.head(10).to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx'

In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Marie-Eve_Analysis")

print("Contenu du projet :")
for p in sorted(base.iterdir()):
    print(p)

Contenu du projet :


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Marie-Eve_Analysis'

In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Marie-Eve_Analysis")

excel_files = list(base.rglob("*.xlsx")) + list(base.rglob("*.xls"))

print("Fichiers Excel trouvés :")
for f in excel_files:
    print(f)

Fichiers Excel trouvés :


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

print("Recherche des fichiers Excel / metadata...\n")

matches = []

for f in root.rglob("*"):
    if f.is_file():
        name = f.name.lower()
        if (
            f.suffix.lower() in [".xlsx", ".xls"]
            or "metadata" in name
            or "meta" in name
        ):
            matches.append(f)

for f in matches:
    print(f)

Recherche des fichiers Excel / metadata...

/content/drive/MyDrive/GWAS results_Gle et Gwi/Synthèse QTLs_Grain length and width.xlsx
/content/drive/MyDrive/GWAS results_Gle et Gwi/Raw GWAS results_Grain width.xlsx
/content/drive/MyDrive/GWAS results_Gle et Gwi/Raw GWAS results_Grain length.xlsx
/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx
/content/drive/MyDrive/Charlotte_DNA_SIP/00_documentation/Metadata pour V1 20251017.xlsx
/content/drive/MyDrive/Charlotte_DNA_SIP/00_documentation/MiSeq run details.xlsx
/content/drive/MyDrive/Charlotte_DNA_SIP/01_metadata/Metadata pour V1 20251017.xlsx
/content/drive/MyDrive/Charlotte_DNA_SIP/01_metadata/metadata_Run3.xlsx
/content/drive/MyDrive/Charlotte_DNA_SIP/01_metadata/metadata_Run5.xlsx
/content/drive/MyDrive/Charlotte_DNA_SIP/01_metadata/metadata_Run3.tsv
/content/drive/MyDrive/Charlotte_DNA_SIP/01_metadata/metadata_Run5.tsv
/content/drive/MyDrive/Charlotte_DNA_SIP/01_metadata/metadata_Run4.tsv
/content/drive/MyDriv

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

print("MyDrive existe :", root.exists())

if root.exists():
    print("\nDossiers ressemblant à Marie-Eve :")
    for p in sorted(root.iterdir()):
        if "marie" in p.name.lower() or "eve" in p.name.lower():
            print(p)

MyDrive existe : True

Dossiers ressemblant à Marie-Eve :
/content/drive/MyDrive/Marie-Eve_Analysis


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

matches = [
    p for p in root.rglob("*")
    if p.is_dir()
    and "marie" in p.name.lower()
    and "analysis" in p.name.lower()
]

print("Dossiers candidats :")
for p in matches:
    print(p)

Dossiers candidats :
/content/drive/MyDrive/Marie-Eve_Analysis


In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Marie-Eve_Analysis")

print("Base existe :", base.exists())

for f in base.rglob("*"):
    if (
        f.is_file()
        and (
            f.suffix.lower() in [".xlsx", ".xls"]
            or "metadata" in f.name.lower()
            or "meta" in f.name.lower()
        )
    ):
        print(f)

Base existe : True
/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx


In [ ]:
import pandas as pd
from pathlib import Path

src = Path("/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx")
out = Path("/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM_qiime2.tsv")

# Lecture
df = pd.read_excel(src)

print("Shape initiale :", df.shape)
print("Colonnes :", list(df.columns))

# Renommer la colonne d'identifiant
df = df.rename(columns={"Échantillon No.": "sample-id"})

# Normaliser les IDs
df["sample-id"] = df["sample-id"].astype(str).str.strip()

# Vérifications
print("\nNombre de lignes :", len(df))
print("IDs uniques :", df["sample-id"].nunique())
print("Doublons :", df["sample-id"].duplicated().sum())

expected = {str(i) for i in range(1, 49)}
observed = set(df["sample-id"])

print("IDs manquants :", sorted(expected - observed, key=int))
print("IDs inattendus :", sorted(observed - expected, key=lambda x: int(x) if x.isdigit() else x))

# Export TSV
# On conserve les valeurs telles quelles, notamment ND
df.to_csv(out, sep="\t", index=False)

print("\nFichier créé :", out)

# Relecture de contrôle
m = pd.read_csv(out, sep="\t", dtype=str)

print("\nShape finale :", m.shape)
print("IDs uniques :", m["sample-id"].nunique())
print("Doublons :", m["sample-id"].duplicated().sum())

print("\nAperçu :")
print(m.head(10).to_string(index=False))

Shape initiale : (48, 18)
Colonnes : ['Échantillon No. ', 'Bloc', 'Cultivar', 'Traitement', 'pH', 'CE', 'NH4', 'NO3', 'P', 'K', 'Ca', 'Mg', 'Fe', 'Cu', 'Mn', 'Zn', 'Mo', 'Al']


KeyError: 'sample-id'

In [ ]:
import pandas as pd
from pathlib import Path

src = Path(
    "/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM.xlsx"
)

out = Path(
    "/content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM_qiime2.tsv"
)

# ============================================================
# 1. Lecture du fichier Excel
# ============================================================

df = pd.read_excel(src)

print("Shape initiale :", df.shape)

print("\nColonnes originales :")
for i, col in enumerate(df.columns):
    print(i, repr(col))

# ============================================================
# 2. Nettoyage des noms de colonnes
# ============================================================

df.columns = [str(c).strip() for c in df.columns]

# La première colonne correspond aux identifiants des échantillons
old_first_column = df.columns[0]

print("\nPremière colonne identifiée :", repr(old_first_column))

df = df.rename(columns={old_first_column: "sample-id"})

# ============================================================
# 3. Normalisation des identifiants
# ============================================================

df["sample-id"] = (
    df["sample-id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# ============================================================
# 4. Vérification des 48 échantillons
# ============================================================

expected = {str(i) for i in range(1, 49)}
observed = set(df["sample-id"])

print("\n===== VALIDATION =====")
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))
print("IDs uniques :", df["sample-id"].nunique())
print("Doublons :", df["sample-id"].duplicated().sum())

missing = expected - observed
unexpected = observed - expected

print(
    "IDs manquants :",
    sorted(missing, key=int) if missing else []
)

print(
    "IDs inattendus :",
    sorted(unexpected) if unexpected else []
)

# ============================================================
# 5. Export QIIME 2 TSV
# ============================================================

# Les données originales sont conservées.
# En particulier, aucune valeur "ND" n'est transformée en zéro.

df.to_csv(
    out,
    sep="\t",
    index=False
)

# ============================================================
# 6. Relecture du fichier produit
# ============================================================

m = pd.read_csv(
    out,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("\n===== FICHIER FINAL =====")
print("Fichier créé :", out)
print("Shape finale :", m.shape)
print("IDs uniques :", m["sample-id"].nunique())
print("Doublons :", m["sample-id"].duplicated().sum())

print("\nColonnes finales :")
print(list(m.columns))

print("\nPremières lignes :")
print(m.head(10).to_string(index=False))

Shape initiale : (48, 18)

Colonnes originales :
0 'Échantillon No. '
1 'Bloc'
2 'Cultivar'
3 'Traitement'
4 'pH'
5 'CE'
6 'NH4'
7 'NO3'
8 'P'
9 'K'
10 'Ca'
11 'Mg'
12 'Fe'
13 'Cu'
14 'Mn'
15 'Zn'
16 'Mo'
17 'Al'

Première colonne identifiée : 'Échantillon No.'

===== VALIDATION =====
Nombre de lignes : 48
Nombre de colonnes : 18
IDs uniques : 48
Doublons : 0
IDs manquants : []
IDs inattendus : []

===== FICHIER FINAL =====
Fichier créé : /content/drive/MyDrive/Marie-Eve_Analysis/01_metadata/Metadata_EM_qiime2.tsv
Shape finale : (48, 18)
IDs uniques : 48
Doublons : 0

Colonnes finales :
['sample-id', 'Bloc', 'Cultivar', 'Traitement', 'pH', 'CE', 'NH4', 'NO3', 'P', 'K', 'Ca', 'Mg', 'Fe', 'Cu', 'Mn', 'Zn', 'Mo', 'Al']

Premières lignes :
sample-id Bloc Cultivar Traitement   pH    CE    NH4    NO3      P       K      Ca      Mg     Fe   Cu    Mn    Zn   Mo      Al
        1    1  Blueray    CTRLMin 3.51 350.3   4.78    0.0  318.1  539.83  898.98  386.24 516.86 1.95 12.58 11.24 4.21 1232.3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BITS="/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS"

find "$BITS" -type f -name "*.fastq.gz" | wc -l
find "$BITS" -type f -name "*.fastq.gz" | sort | head -12

SyntaxError: invalid syntax (3320869032.py, line 3)

In [ ]:
cd /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq

tar -czf BITS_fastq.tar.gz BITS

ls -lh BITS_fastq.tar.gz

SyntaxError: invalid decimal literal (986128809.py, line 1)

In [ ]:
!find "/content/drive/MyDrive/Marie-Eve_Analysis" \
  -maxdepth 5 \
  -type d \
  -iname "*BITS*" \
  -print

/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS
/content/drive/MyDrive/Marie-Eve_Analysis/04_work/fastqc/BITS
/content/drive/MyDrive/Marie-Eve_Analysis/05_results/multiqc_BITS


In [ ]:
!find "/content/drive/MyDrive/Marie-Eve_Analysis" \
  -maxdepth 5 \
  -type f \
  \( -name "*.fastq.gz" -o -name "*.fq.gz" \) \
  | grep -i BITS \
  | sort \
  | head -30

/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/10BITSLEM26-200_S1_L001_R1_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/10BITSLEM26-200_S1_L001_R2_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/11BITSLEM26-200_S1_L001_R1_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/11BITSLEM26-200_S1_L001_R2_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/12BITSLEM26-200_S1_L001_R1_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/12BITSLEM26-200_S1_L001_R2_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/13BITSLEM26-200_S1_L001_R1_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/13BITSLEM26-200_S1_L001_R2_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/14BITSLEM26-200_S1_L001_R1_001.fastq.gz
/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS/14BITSLEM26-200_S1_L001_R2_001.fastq.gz


In [ ]:
!find "/content/drive/MyDrive/Marie-Eve_Analysis" \
  -maxdepth 5 \
  -type f \
  \( -name "*.fastq.gz" -o -name "*.fq.gz" \) \
  | grep -i BITS \
  | wc -l

96


In [ ]:
!cd "/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq" && \
tar -czf BITS_fastq.tar.gz BITS

In [ ]:
!ls -lh "/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS_fastq.tar.gz"

-rw------- 1 root root 831M Oct  9 14:20 /content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS_fastq.tar.gz


In [ ]:
!tar -tzf "/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS_fastq.tar.gz" | head -20

BITS/
BITS/10BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/10BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/11BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/11BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/12BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/12BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/13BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/13BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/14BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/14BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/15BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/15BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/16BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/16BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/17BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/17BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/18BITSLEM26-200_S1_L001_R1_001.fastq.gz
BITS/18BITSLEM26-200_S1_L001_R2_001.fastq.gz
BITS/19BITSLEM26-200_S1_L001_R1_001.fastq.gz


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/Marie-Eve_Analysis/02_raw_fastq/BITS_fastq.tar.gz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>